# Flight Dynamics Model

Derive the discrete rigid-body map and its hover linearization with SymPy.

## Analytic vs numerical: what blows up

A compact flight model is a short list of **matrix** operations. SymPy becomes huge when those operations are **substituted into every scalar entry** of a $13\times 1$ vector. The physics did not grow; the algebra did.

Operations that explode a symbolic expression (avoid expanding them analytically; evaluate them numerically before or after the step):

| Operation | Why it explodes | What to do instead |
|---|---|---|
| Generic matrix inverse `M.inv()` | Adjugate / determinant fills every entry with rationals | Supply $I^{-1}$ (here a diagonal) as an input constant; for rotations use $R^{\mathsf{T}}$ when $R$ is orthogonal |
| Quaternion `normalize()` | Divides by $\sqrt{q\cdot q}$ and pulls every force/rate term under that root | Assume unit $q$ into the step; renormalize **numerically after** the update |
| Semi-implicit $q\leftarrow q(\omega^+)$ | Pastes the already-expanded $\omega^+$ (hence $I^{-1}$ and the torques) into all four quaternion components | Explicit Euler: integrate $q$ with the **start-of-step** $\omega$ |
| Printing the stacked $13\times 1$ after substitution | Inlines $R(q)$, $I^{-1}$, and products into every row | Keep the **block** form ($\tau$, $\omega^+$, $q^+$, $v^+$, $p^+$) as the model |

Rule of thumb: **inverses, norms, and square roots** belong in a numerical kernel (or as precomputed constants). The symbolic notebook should keep multiplies, adds, and cross products on matrices.

## Model Constants

**Assumption (airframe):** products of inertia are zero, so $I = \mathrm{diag}(I_{xx}, I_{yy}, I_{zz})$.

**Simplification (analytic):** the notebook takes $I^{-1} = \mathrm{diag}(1/I_{xx}, 1/I_{yy}, 1/I_{zz})$ as a constant (`II_inv`). Do not call a generic symbolic `II.inv()` in the equations of motion.

**Rotors:** $k_t$ and $k_m$ are constants. The step computes thrust and the shaft drag moment from the rotor speed. They are not inputs.

In [ ]:
from sympy import Matrix, Quaternion, Symbol, sqrt, symbols

# Constants
m = symbols("m", real=True)  # mass
g = symbols("g", real=True)  # gravity
world_gravity = Matrix([0, 0, -g])
# Principal-axis inertia (products of inertia are zero on this airframe).
I_xx, I_yy, I_zz = symbols("I_xx I_yy I_zz", real=True, positive=True)
II = Matrix.diag(I_xx, I_yy, I_zz)
II_inv = Matrix.diag(1 / I_xx, 1 / I_yy, 1 / I_zz)
dt = symbols("dt", real=True)  # time step

# Relative positions of the rotors in the body frame from the center of mass
r1x, r1y, r1z = symbols("r1x r1y r1z", real=True)  # vector from the center of mass to the first rotor
r2x, r2y, r2z = symbols("r2x r2y r2z", real=True)  # vector from the center of mass to the second rotor
r3x, r3y, r3z = symbols("r3x r3y r3z", real=True)  # vector from the center of mass to the third rotor
r4x, r4y, r4z = symbols("r4x r4y r4z", real=True)  # vector from the center of mass to the fourth rotor
r1 = Matrix([r1x, r1y, r1z])
r2 = Matrix([r2x, r2y, r2z])
r3 = Matrix([r3x, r3y, r3z])
r4 = Matrix([r4x, r4y, r4z])

# Spin sign about body +z, matching Gazebo turningDirection: +1 CCW, -1 CW.
s1, s2, s3, s4 = symbols("s1 s2 s3 s4", real=True)

# k_t: thrust per speed squared, N/(rad/s)^2 (Gazebo motorConstant).
# k_m: shaft drag moment per thrust, metres (Gazebo momentConstant). M = k_m * T.
k_t, k_m = symbols("k_t k_m", real=True)

## State and Input Space of Model

State $x = (p_w,\, v_b,\, q,\, \omega_b)\in\mathbb{R}^{13}$, input $u = (\omega_1,\omega_2,\omega_3,\omega_4)$, the commanded rotor speeds in rad/s.

Each speed becomes a thrust and a shaft drag moment inside the step:

$$
T_i = k_t \omega_i^2, \qquad F_i = (0,\, 0,\, T_i), \qquad M_i = k_m T_i.
$$

$M_i$ is the drag moment about the rotor shaft. The letter $M$ is used so it is not confused with the quaternion $q$. On the X3, $k_t$ is Gazebo's `motorConstant` and $k_m$ is `momentConstant`.

The spin sign $s_i$ matches Gazebo `turningDirection`: $+1$ counterclockwise about body $+z$, $-1$ clockwise. Rotors 0 and 1 are counterclockwise, rotors 2 and 3 clockwise. The reaction on the body is $\tau_{z,i} = -s_i M_i$, so a counterclockwise rotor yaws the aircraft clockwise.

**Numerical assumption:** $q$ entering each step is a **unit** quaternion. The symbolic map does not renormalize; do that after the update when you simulate.

In [ ]:
# Rotor forces pointing up in the body frame (input)
omega1, omega2, omega3, omega4 = symbols("omega1 omega2 omega3 omega4", real=True)  # rotor angular velocities
input_vector = Matrix([omega1, omega2, omega3, omega4])

# Translational state space
x, y, z = symbols("x y z", real=True)  # world position
world_position = Matrix([x, y, z])
v_x, v_y, v_z = symbols("v_x v_y v_z", real=True)  # body velocity
body_velocity = Matrix([v_x, v_y, v_z])

# Rotational state space using Quaternions instead of euler angles
q0, q1, q2, q3 = symbols("q0 q1 q2 q3", real=True)  # world orientation
q = Quaternion(q0, q1, q2, q3)
w_x, w_y, w_z = symbols("w_x w_y w_z", real=True)  # body angular velocity
body_angular_velocity = Matrix([w_x, w_y, w_z])

state_vector = Matrix([world_position, body_velocity, q.to_Matrix(), body_angular_velocity])

## Physical Equations of Motion

Discrete forward-Euler step of the rigid body. The cell below builds the block map; it does **not** expand a generic matrix inverse or a quaternion norm.

**Assumptions baked into this cell**

- Unit quaternion into the step ($\|q\|=1$), so $R(q)^{-1} = R(q)^{\mathsf{T}}$.
- Principal-axis inertia; multiply rates by the precomputed `II_inv`.
- Explicit Euler on attitude: $\dot q = \tfrac12 q \otimes \omega$ uses the **current** $\omega$, not $\omega^+$.
- Thrust $T_i = k_t \omega_i^2$ and shaft drag moment $M_i = k_m T_i$. Body yaw from that moment is $\tau_{z,i} = -s_i M_i$.
- Cross-flow is left out. *Querströmung* is the airflow perpendicular to the rotor shaft. Gazebo models it with `rotorDragCoefficient` (an in-plane force) and `rollingMomentCoefficient` (a rolling moment), both proportional to that airspeed and to rotor speed. Neither term is in this map.
- Angular acceleration of the rotor is neglected. Usually, this would also create an additional torque on the body. Therefore, the angular velocity of the rotors is assumed to be constant and changes are instantaneous.

**Numerical step after this map (not in SymPy):** renormalize $q \leftarrow q/\|q\|$ so the next step still sees a unit attitude.

In [ ]:
# For a unit quaternion, R^{-1} = R^T. Keep R as a Matrix; do not expand .inv().
def rotation_matrix() -> Matrix:
    return q.to_rotation_matrix()


def world_to_body(world_vec: Matrix) -> Matrix:
    return rotation_matrix() * world_vec


def body_to_world(body_vec: Matrix) -> Matrix:
    return rotation_matrix().T * body_vec


def torque_of_rotor_thrust_on_body(rotor_force: Matrix, rotor_position: Matrix) -> Matrix:
    return rotor_position.cross(rotor_force)


def rotor_thrust_and_drag_moment(rotor_angular_vel: Symbol) -> tuple[Symbol, Symbol]:
    thrust = k_t * rotor_angular_vel**2
    drag_moment = k_m * thrust
    return thrust, drag_moment


def torque_of_rotor_drag_moment_on_body(rotor_drag_moment: Symbol, rotor_turn_direction: Symbol) -> Matrix:
    # s = +1 CCW, -1 CW. The leading minus is the reaction on the body, as in Gazebo.
    body_z_axis = Matrix([0, 0, 1])
    return -rotor_drag_moment * rotor_turn_direction * body_z_axis


def torque_to_angular_acceleration(torque: Matrix, angular_velocity: Matrix) -> Matrix:
    # Euler: I^{-1} (tau - omega x I omega), with principal-axis I^{-1} already diagonal.
    angular_momentum = II * angular_velocity
    gyroscopic_torque = angular_velocity.cross(angular_momentum)  # omega x (I @ omega)
    return II_inv * (torque - gyroscopic_torque)


def apply_angular_velocity_to_quaternion(omega: Matrix, attitude: Quaternion) -> Quaternion:
    # Explicit Euler of q_dot = 1/2 q ⊗ omega. Renormalize numerically after the step.
    omega_quat = Quaternion(0, omega[0], omega[1], omega[2])
    return attitude + attitude * omega_quat * dt / 2


# Thrust T and shaft drag moment M for each rotor. M is not a quaternion component.
t1, M1 = rotor_thrust_and_drag_moment(omega1)
t2, M2 = rotor_thrust_and_drag_moment(omega2)
t3, M3 = rotor_thrust_and_drag_moment(omega3)
t4, M4 = rotor_thrust_and_drag_moment(omega4)

# In body frame, the thrust is pointing up, so the force is [0, 0, t]
F1 = Matrix([0, 0, t1])
F2 = Matrix([0, 0, t2])
F3 = Matrix([0, 0, t3])
F4 = Matrix([0, 0, t4])

# Torque caused by thrusts of rotors along the arms (around the body x- and y-axes)
body_total_thrust_torque = (
    torque_of_rotor_thrust_on_body(F1, r1)
    + torque_of_rotor_thrust_on_body(F2, r2)
    + torque_of_rotor_thrust_on_body(F3, r3)
    + torque_of_rotor_thrust_on_body(F4, r4)
)

# Torque caused by drag moments of rotors (around the body z-axis)
body_total_drag_moment_torque = (
    torque_of_rotor_drag_moment_on_body(M1, s1)
    + torque_of_rotor_drag_moment_on_body(M2, s2)
    + torque_of_rotor_drag_moment_on_body(M3, s3)
    + torque_of_rotor_drag_moment_on_body(M4, s4)
)

body_total_torque = body_total_thrust_torque + body_total_drag_moment_torque

body_angular_acceleration = torque_to_angular_acceleration(body_total_torque, body_angular_velocity)
new_body_angular_velocity = body_angular_velocity + body_angular_acceleration * dt

# Attitude uses the start-of-step rate (explicit Euler of the continuous kinematics).
new_q = apply_angular_velocity_to_quaternion(body_angular_velocity, q)

# Translational Dynamics
body_gravity = world_to_body(world_gravity)  # g
body_gravitational_force = m * body_gravity  # g*m
body_net_force = body_gravitational_force + F1 + F2 + F3 + F4
body_acceleration = body_net_force / m - body_angular_velocity.cross(body_velocity)
new_body_velocity = body_velocity + body_acceleration * dt
new_world_velocity = body_to_world(new_body_velocity)
new_world_position = world_position + new_world_velocity * dt

# Discrete nonlinear map x_{k+1} = f(x_k, u_k), kept as matrix blocks (not fully expanded).
new_state_vector = Matrix([new_world_position, new_body_velocity, new_q.to_Matrix(), new_body_angular_velocity])

## Nonlinear state transfer

One step is $x_{k+1} = f(x_k, u_k)$ with $x = (p,\, v,\, q,\, \omega)$ and $u = (\omega_1, \omega_2, \omega_3, \omega_4)$. With $T_i = k_t \omega_i^2$, $F_i = (0,\, 0,\, T_i)$, and $M_i = k_m T_i$ the **compact** form is

$$
\begin{aligned}
\tau &= \sum_{i=1}^{4} r_i \times F_i - \sum_{i=1}^{4} s_i M_i\, e_z, \\
\omega^+ &= \omega + \Delta t\, I^{-1}\big(\tau - \omega \times I\omega\big), \\
q^+ &= q + \tfrac{1}{2}\Delta t\, q \otimes \omega, \\
v^+ &= v + \Delta t\left(R(q)\, g_w + \tfrac{1}{m}\sum_{i=1}^{4} F_i\right), \\
p^+ &= p + \Delta t\, R(q)^{\mathsf{T}} v^+.
\end{aligned}
$$

$e_z = (0, 0, 1)$. The spin sign is $s_i = +1$ counterclockwise and $s_i = -1$ clockwise, matching Gazebo. The second sum is the shaft reaction that yaws the body: a counterclockwise rotor contributes a negative (clockwise) yaw moment.

**Simplifications already applied (same continuous physics, cleaner discrete algebra)**

1. $I^{-1}$ is an **input constant** (diagonal), not `II.inv()` inside $f$.
2. $R^{\mathsf{T}}$ replaces $R^{-1}$ under the unit-quaternion assumption.
3. No symbolic `normalize`; assume $\|q\|=1$ on input, renormalize **numerically after** $f$.
4. Attitude uses start-of-step $\omega$ (explicit Euler), so $\omega^+$ is not substituted into $q^+$.
5. Cross-flow (*Querströmung*) is omitted. That is the airspeed perpendicular to the shaft. Gazebo's `rotorDragCoefficient` turns it into an in-plane force, and `rollingMomentCoefficient` turns it into a rolling moment. Both are neglected for now.

**Further simplifications you could still make (optional)**

- Leave $R(q)$ as a matrix symbol and never expand `to_rotation_matrix()` until code generation / the GPU kernel.
- Drop $\omega \times I\omega$ when rates stay small (near-hover); that only removes gyroscopic coupling.
- Take the control wrench $(F_z,\, \tau)$ as the input instead of four rotor speeds (moves allocation outside $f$).
- Prefer SymPy `cse()` / block evaluation over printing the stacked $13\times 1$.

In [ ]:
# Discrete nonlinear transfer x_{k+1} = f(x_k, u_k) as the stacked block map.
# Prefer the CSE cell below for a readable printout (avoids re-printing every repeated R_ij).
nonlinear_f = new_state_vector

## CSE form of the nonlinear map

Common subexpression elimination (`sympy.cse`) does not change the physics. It walks the expression tree for $f(x,u)$ and pulls repeated pieces into temporaries $t_0, t_1, \ldots$ (for example entries of $R(q)$, or shared force sums). The cell below prints those temporaries and then the reduced $13\times 1$ that uses them—the same map as `nonlinear_f`, cheaper to read and closer to what a code generator would emit.

In [ ]:
from sympy import cse, numbered_symbols

# Temporaries + reduced expressions for x_{k+1} = f(x_k, u_k).
cse_replacements, cse_reduced = cse(list(nonlinear_f), symbols=numbered_symbols("t"))
nonlinear_f_cse = Matrix(cse_reduced)

print(f"{len(cse_replacements)} common subexpressions\n")
for symbol, expr in cse_replacements:
    print(f"{symbol} = {expr}")
print("\nreduced f(x, u) =")
nonlinear_f_cse

## Jacobian linearization about hover

Deviations from hover $(x_\star, u_\star)$, written $\delta x = x - x_\star$ and $\delta u = u - u_\star$, obey the linear state transfer

$$
\delta x_{k+1} = (I + \Delta t\, A)\, \delta x_k + (\Delta t\, B)\, \delta u_k
$$

$$
A = \frac{\partial \dot x}{\partial x}\Big|_{x_\star, u_\star}, \quad
B = \frac{\partial \dot x}{\partial u}\Big|_{x_\star, u_\star}, \quad
\dot x = \frac{f - x}{\Delta t}.
$$

**Assumptions:** same discrete $f$ as above (unit $q$, principal-axis $I^{-1}$, no symbolic normalize, explicit Euler on $q$, shaft drag moment included, cross-flow omitted). Hover trim is level attitude, zero velocity and rate, each rotor at $\omega = \sqrt{mg/(4 k_t)}$ so each thrust is a quarter of the weight. Opposite spin signs cancel the yaw couples at that common speed. `state_matrix` is the $13 \times 17$ block $[I + \Delta t\, A \mid \Delta t\, B]$ — a local linear map on **deviations**, not a global replacement for nonlinear $f$.

In [ ]:
# state_rate = (f - x) / dt, then A = d(state_rate)/dx and B = d(state_rate)/du at hover.
# (Named state_rate so it does not shadow the gravity symbol g.)
state_rate = (nonlinear_f - state_vector) / dt
A = state_rate.jacobian(state_vector)
B = state_rate.jacobian(input_vector)

# Hover: level attitude, no motion, each rotor at the speed whose thrust is mg/4.
omega_hover = sqrt(m * g / (4 * k_t))
hover = {
    v_x: 0,
    v_y: 0,
    v_z: 0,
    w_x: 0,
    w_y: 0,
    w_z: 0,
    q0: 1,
    q1: 0,
    q2: 0,
    q3: 0,
    omega1: omega_hover,
    omega2: omega_hover,
    omega3: omega_hover,
    omega4: omega_hover,
}
A_hover = A.subs(hover).doit()
B_hover = B.subs(hover).doit()
A_hover.simplify()
B_hover.simplify()

# Multiplies the stacked deviation (dx, du): dx' = dx + dt * (A dx + B du)
state_matrix = (Matrix.eye(A_hover.rows) + dt * A_hover).row_join(dt * B_hover)
state_matrix

## Jacobian linearization about the current state (small $\Delta t$)

Hover froze $(A,B)$ at one trim $(x_\star,u_\star)$. If $\Delta t$ is small and you only care about a **small deviation** $(\delta x_k,\delta u_k)$ from whatever state you are in now, linearize **at the current** $(x_k,u_k)$ instead:

$$
A_k = \frac{\partial \dot x}{\partial x}\Big|_{x_k,u_k}, \quad
B_k = \frac{\partial \dot x}{\partial u}\Big|_{x_k,u_k}, \quad
\dot x = \frac{f-x}{\Delta t},
$$

$$
\delta x_{k+1} \approx (I + \Delta t\, A_k)\,\delta x_k + (\Delta t\, B_k)\,\delta u_k.
$$

**Assumptions:** same discrete $f$ as above; $\Delta t$ small enough that one Euler step matches the continuous Jacobian; $\delta x$ small enough that the first-order model is honest locally. $A_k$ and $B_k$ stay **functions of** $(x,u)$—re-evaluate them each step (gain scheduling / EKF-style). They are not replaced by the hover numbers.

The cell below builds those symbolic Jacobians and the discrete block $[I+\Delta t A_k \mid \Delta t B_k]$, then prints a CSE form so the state dependence is readable.

In [ ]:
# Linearize about the *current* symbolic (x, u), not the hover trim.
# Reuse state_rate from the hover cell if present; otherwise rebuild.
state_rate = (nonlinear_f - state_vector) / dt
A_k = state_rate.jacobian(state_vector)
B_k = state_rate.jacobian(input_vector)

# Discrete LTV map on a small deviation over one small step:
#   delta_x^{+} ≈ (I + dt A_k) delta_x + (dt B_k) delta_u
state_matrix_k = (Matrix.eye(A_k.rows) + dt * A_k).row_join(dt * B_k)

cse_k_replacements, cse_k_reduced = cse(list(state_matrix_k), symbols=numbered_symbols("s"))
state_matrix_k_cse = Matrix(cse_k_reduced).reshape(state_matrix_k.rows, state_matrix_k.cols)

print(
    f"A_k is {A_k.rows}x{A_k.cols}, B_k is {B_k.rows}x{B_k.cols}, "
    f"state_matrix_k is {state_matrix_k.rows}x{state_matrix_k.cols} "
    f"({len(cse_k_replacements)} common subexpressions)\n"
)
for symbol, expr in cse_k_replacements:
    print(f"{symbol} = {expr}")
print("\nreduced [I + dt A_k | dt B_k] =")
state_matrix_k_cse

## NumPy callable for the current-state linearization

`lambdify` turns the symbolic $A_k(x,u)$ and $B_k(x,u)$ into a NumPy function (with CSE in the generated body). Arguments are the full state, the four rotor speeds, then the model constants ($m$, $g$, $\Delta t$, inertia, rotor positions, $k_t$, $k_m$, spin signs). The return is `(A_k, B_k)`; the discrete maps are $I + \Delta t A_k$ and $\Delta t B_k$.

In [ ]:
from sympy import lambdify

# Order matches state_vector / input_vector, then constants the Jacobians depend on.
linearization_args = (
    # state x
    x,
    y,
    z,
    v_x,
    v_y,
    v_z,
    q0,
    q1,
    q2,
    q3,
    w_x,
    w_y,
    w_z,
    # input u: rotor speeds (rad/s)
    omega1,
    omega2,
    omega3,
    omega4,
    # constants
    m,
    g,
    dt,
    I_xx,
    I_yy,
    I_zz,
    r1x,
    r1y,
    r1z,
    r2x,
    r2y,
    r2z,
    r3x,
    r3y,
    r3z,
    r4x,
    r4y,
    r4z,
    k_t,
    k_m,
    s1,
    s2,
    s3,
    s4,
)

current_state_linearization_np = lambdify(
    linearization_args,
    (A_k, B_k),
    modules="numpy",
    cse=True,
    docstring_limit=None,  # keep full generated source in __doc__
)

# Full NumPy function body (CSE already applied by lambdify).
print(current_state_linearization_np.__doc__)

## Evaluation: batched runtime and linearization error

Microbenchmark of the discrete map and its current-state Jacobians. Variants:

| Name | What is timed |
|---|---|
| `numpy_nonlinear` / `torch_nonlinear` | Batched $x\leftarrow f(x,u)$ for $T$ steps |
| `numpy_linear_map` / `torch_linear_map` | Batched $\delta x\leftarrow (I+\Delta t A)\delta x+(\Delta t B)\delta u$ with $A,B$ frozen per item |
| `numpy_jacobian_analytic` | SymPy-`lambdify` $A_k,B_k$ (rate Jacobians) per item |
| `torch_jacobian_analytic` | Same formula, plus copy into Torch tensors |
| `torch_jacobian_autograd` | `torch.autograd.functional.jacobian` of the discrete Torch $f$ |

**Also reported:** analytic vs autograd Jacobian disagreement; one-step linear prediction error vs nonlinear $f$; NumPy vs Torch nonlinear agreement. Helpers live in `dynamics_benchmark.py` (same folder). Batch sizes vary; Jacobian timings use fewer inner steps at large $N$ so the cell finishes in reasonable time. CPU only unless you change the Torch device.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Notebook cwd may be repo root or control/; make the helper importable either way.
_notebook_dir = Path.cwd() / "control" / "notebooks"
if not (_notebook_dir / "dynamics_benchmark.py").exists():
    _notebook_dir = Path.cwd()
if not (_notebook_dir / "dynamics_benchmark.py").exists():
    _notebook_dir = Path.cwd() / "notebooks"
sys.path.insert(0, str(_notebook_dir.resolve()))

from dynamics_benchmark import (  # noqa: E402
    X3,
    hover_speed,
    measure_errors,
    results_to_arrays,
    run_timing_suite,
    step_nonlinear_np,
)

_const = (
    float(X3["m"]),
    float(X3["g"]),
    float(X3["dt"]),
    float(X3["I_xx"]),
    float(X3["I_yy"]),
    float(X3["I_zz"]),
    *X3["r"].reshape(-1).tolist(),
    float(X3["k_t"]),
    float(X3["k_m"]),
    *X3["s"].tolist(),
)


def jac_analytic_np(x_vec: np.ndarray, u_vec: np.ndarray):
    """Rate Jacobians A=d(state_rate)/dx, B=d(state_rate)/du from SymPy lambdify."""
    return current_state_linearization_np(*x_vec.tolist(), *u_vec.tolist(), *_const)


nonlinear_f_np = lambdify(
    linearization_args,
    nonlinear_f,
    modules="numpy",
    cse=True,
)

_x0 = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
_u0 = np.full(4, hover_speed())
_f_sym = np.asarray(nonlinear_f_np(*_x0.tolist(), *_u0.tolist(), *_const), dtype=float).reshape(13)
_f_np = step_nonlinear_np(_x0, _u0)[0]
print("hover |f_sym - f_np|_inf =", float(np.max(np.abs(_f_sym - _f_np))))

In [ ]:
BATCH_SIZES = [1, 8, 32, 128, 512]
STEPS = 60  # propagation steps; Jacobian variants use fewer inner steps at large N

bench_results = run_timing_suite(jac_analytic_np, BATCH_SIZES, steps=STEPS)
error_metrics = measure_errors(jac_analytic_np)

print("Error metrics")
for key, value in error_metrics.items():
    print(f"  {key}: {value:.6e}" if isinstance(value, float) else f"  {key}: {value}")

print("\nTiming (us per step·item, lower is better)")
print(f"{'variant':28} {'N':>6} {'steps':>6} {'us/item':>12} {'Hz·item':>12}")
for row in bench_results:
    print(f"{row.name:28} {row.batch_size:6d} {row.steps:6d} {row.us_per_step_item:12.3f} {row.steps_per_sec:12.1f}")

In [ ]:
series = results_to_arrays(bench_results)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)

# Cost vs batch size
for name, data in series.items():
    axes[0].plot(data["batch_sizes"], data["us_per_step_item"], marker="o", label=name)
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel("batch size N")
axes[0].set_ylabel("µs per (step · item)")
axes[0].set_title("Compute cost vs batch size")
axes[0].grid(True, which="both", alpha=0.3)
axes[0].legend(fontsize=8)

# Throughput for dynamics only (exclude jacobian variants for readability)
for name, data in series.items():
    if "jacobian" in name:
        continue
    axes[1].plot(data["batch_sizes"], data["steps_per_sec"], marker="o", label=name)
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_xlabel("batch size N")
axes[1].set_ylabel("step·item / s")
axes[1].set_title("Throughput (propagation maps)")
axes[1].grid(True, which="both", alpha=0.3)
axes[1].legend(fontsize=8)

fig.suptitle("Flight dynamics microbenchmark (CPU)")
_fig_path = _notebook_dir / "dynamics_benchmark_results.png"
fig.savefig(_fig_path, dpi=140)
print("wrote", _fig_path)
plt.show()

# Practical summary at largest batch
n_max = max(BATCH_SIZES)
at_max = [r for r in bench_results if r.batch_size == n_max]
fastest = min(at_max, key=lambda r: r.us_per_step_item)
cheapest_nl = min(
    (r for r in at_max if r.name.endswith("nonlinear")),
    key=lambda r: r.us_per_step_item,
)
cheapest_jac = min(
    (r for r in at_max if "jacobian" in r.name),
    key=lambda r: r.us_per_step_item,
)
print(f"\nAt N={n_max}:")
print(f"  lowest µs/item overall: {fastest.name} ({fastest.us_per_step_item:.3f})")
print(f"  fastest nonlinear step: {cheapest_nl.name} ({cheapest_nl.us_per_step_item:.3f})")
print(f"  cheapest Jacobian:      {cheapest_jac.name} ({cheapest_jac.us_per_step_item:.3f})")
print(
    "  analytic vs autograd Jac max ‖·‖:",
    f"{error_metrics['jac_analytic_vs_autograd_max']:.3e}",
)
print(
    "  linear one-step err (mean, δ~1e-4):",
    f"{error_metrics['linear_onestep_err_mean']:.3e}",
)

## Teaching notes (common questions)

### Why quaternions, not Euler angles or a raw rotation matrix?

Three common attitude charts:

| Representation | Degrees of freedom stored | Main failure mode |
|---|---|---|
| Euler / Tait–Bryan angles (roll–pitch–yaw) | 3 | **Gimbal lock**: two axes align, one degree of freedom disappears, rates blow up in that chart |
| Rotation matrix $R\in\mathrm{SO}(3)$ | 9 (with 6 constraints) | No gimbal lock; must keep $R^{\mathsf{T}}R=I$ and $\det R=1$ under integration noise |
| Unit quaternion $q$ | 4 (with 1 constraint $\|q\|=1$) | No gimbal lock; must renormalize; double cover ($q$ and $-q$ are the same attitude) |

**Euler angles** are fine for *displaying* a hover attitude. They are a poor *state* for integrating a tumbling aircraft: near pitch $\pm 90^\circ$ the chart singularities make $\dot\theta$ formulas explode even though the physical motion is smooth. That is the gimbal-lock risk, and for a quadrotor that can flip it is significant.

**Rotation matrices** are a perfectly valid state. Kinematics are $\dot R = R[\omega]_\times$ (body rate). There is no gimbal lock. The cost is nine numbers and the need to project back onto $\mathrm{SO}(3)$ after each step (e.g. SVD / Gram–Schmidt), which is heavier than renormalizing a quaternion. Controllers written "on $\mathrm{SO}(3)$" often use $R$ for the *error*, even when the estimator stores $q$.

**Unit quaternions** are the usual compromise in flight software: four numbers, smooth kinematics $\dot q = \tfrac12 q\otimes\omega$, cheap renormalization, no chart singularity. This notebook uses them for that reason—not because matrices are wrong.

### What would `cse()` change compared to this notebook?

Nothing about the *physics*. Common subexpression elimination (`sympy.cse`) looks at a large expression tree and pulls out repeated pieces into temporaries, e.g.

$$
\begin{aligned}
t_0 &= R(q), \\
t_1 &= I^{-1}(\tau - \omega\times I\omega), \\
\omega^+ &= \omega + \Delta t\, t_1, \\
\ldots
\end{aligned}
$$

**Today** we keep the model as **named blocks** (`tau`, `new_body_angular_velocity`, …). That is already human-level CSE: we refuse to inline everything into one $13\times 1$ printout.

**`cse()`** helps when you *do* expand (code generation for C++/CUDA, or a fully substituted Jacobian) and SymPy would otherwise recompute the same $R_{ij}$ or force sum dozens of times. It returns `(replacements, reduced_exprs)`. It does **not** replace $R^{-1}$ by $R^{\mathsf{T}}$ or remove `normalize`; those are modeling choices, not CSE.

### Can we linearize about the *current* state every step if $\Delta t$ is small?

Yes. That is a standard idea under several names: **gain scheduling**, **LTV (linear time-varying) model**, or in estimation the **EKF** linearization.

Hover linearization (this notebook) freezes

$$
A = \frac{\partial f}{\partial x}\Big|_{x_\star,u_\star}, \quad
B = \frac{\partial f}{\partial u}\Big|_{x_\star,u_\star}
$$

at one trim. If instead, at each step $k$, you set

$$
A_k = \frac{\partial f}{\partial x}\Big|_{x_k,u_k}, \quad
B_k = \frac{\partial f}{\partial u}\Big|_{x_k,u_k},
$$

then for a **small** deviation $\delta x_k$ over a **small** $\Delta t$,

$$
\delta x_{k+1} \approx (I + \Delta t\, A_k)\,\delta x_k + (\Delta t\, B_k)\,\delta u_k.
$$

Small $\Delta t$ keeps the discrete Euler consistent with the continuous Jacobian; small attitude error keeps the first-order (affine) model honest *locally*. It does **not** make one constant matrix valid for large flips—$A_k$ must be **recomputed** (or looked up) as the aircraft moves.

Practical consequences:

- **Control:** an LQR gain $K(x_k)$ scheduled on the current attitude/velocity, or a nonlinear design that never linearizes (geometric / INDI).
- **Estimation:** this is what an EKF does with $F_k = \partial f/\partial x$ at the current estimate.
- **Cost:** you need $A_k,B_k$ every step (analytic Jacobian as here, or automatic / finite differences). Hover $A,B$ stay cheap and constant; scheduled $A_k,B_k$ are more accurate away from hover and more work.

So: small $\Delta t$ justifies *local* linearization along the trajectory; it is not an excuse to keep the *hover* matrix when the aircraft is inverted.